# 킥보드 v3 학습 (Colab T4)
5클래스(kickboard, person, car, motorcycle, animal), `train_v2_s09` 가중치에서 시작, scale 0.9, 30 epoch

**준비**: 구글 드라이브 `내 드라이브/datset/`에 백업 zip들이 있어야 함
**런타임**: 메뉴 > 런타임 > 런타임 유형 변경 > T4 GPU

세션이 끊기면 1~3번을 다시 실행한 뒤 **5번(이어서 학습)**만 실행하면 됨. 체크포인트는 드라이브 `yolo_runs/`에 저장됨

In [ ]:
# 1. GPU 확인, 드라이브 연결, ultralytics 설치 (PC와 같은 버전)
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.177

In [ ]:
# 2. 백업 zip을 Colab 로컬 디스크로 복사해서 풀기 (드라이브에서 바로 읽으면 학습이 매우 느림)
import glob, os, zipfile
BACKUP = '/content/drive/MyDrive/datset'
PROJ = '/content/proj'
os.makedirs(PROJ, exist_ok=True)
zips = sorted(glob.glob(f'{BACKUP}/*.zip'))
assert zips, f'{BACKUP}에 zip이 없음 - 드라이브 업로드 위치 확인'
for z in zips:
    print('풀기:', os.path.basename(z))
    zipfile.ZipFile(z).extractall(PROJ)
# Colab 터미널에서 cd /content/proj && claude 로 실행하면 이 두 파일을 읽고 맥락을 잡음
import shutil
for f in ('CLAUDE.md', 'HANDOFF.md'):
    if os.path.exists(f'{BACKUP}/{f}'):
        shutil.copy(f'{BACKUP}/{f}', PROJ)
!du -sh /content/proj/datasets/*

In [ ]:
# 3. v3 폴더 정리: 백업엔 v3 라벨만 있음 → 원본 이미지를 v3 폴더에 링크, yaml 경로를 Colab 경로로
from pathlib import Path
D = Path(PROJ) / 'datasets'
V3 = D / 'v3'
for lbl_dir in sorted(V3.rglob('labels')):
    split = lbl_dir.parent.relative_to(V3)
    src, dst = D / split / 'images', V3 / split / 'images'
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for img in src.iterdir():
        if not (dst / img.name).exists():
            os.link(img, dst / img.name)
        n += 1
    print(f'{split}: 이미지 {n}장, 라벨 {len(list(lbl_dir.glob("*.txt")))}개')

yaml_text = (D / 'v3.yaml').read_text()
(D / 'v3_colab.yaml').write_text(yaml_text)
print(yaml_text)

In [ ]:
# 4. 학습 (처음 한 번만). 결과·체크포인트는 드라이브에 저장돼서 세션이 끊겨도 남음
from ultralytics import YOLO
RUNS = '/content/drive/MyDrive/yolo_runs'
model = YOLO(f'{PROJ}/runs/kickboard/train_v2_s09/weights/best.pt')
model.train(
    data=str(D / 'v3_colab.yaml'),
    epochs=30, imgsz=640, scale=0.9,
    batch=32,          # PC는 3GB라 8이었음. T4 16GB라 32
    workers=2,         # Colab 무료는 CPU 2개
    cache='disk',      # 첫 epoch 전에 이미지를 미리 풀어둬서 데이터 읽기 병목을 줄임
    project=RUNS, name='train_v3_5cls', exist_ok=True,
)

In [ ]:
# 5. 세션이 끊긴 뒤 이어서 학습 (1~3번 실행 후 이것만)
from ultralytics import YOLO
RUNS = '/content/drive/MyDrive/yolo_runs'
YOLO(f'{RUNS}/train_v3_5cls/weights/last.pt').train(resume=True)

In [ ]:
# 6. test 셋 평가 (킥보드 test만 정답 라벨, 나머지 클래스는 자동 라벨 기준이라 참고용)
from ultralytics import YOLO
m = YOLO('/content/drive/MyDrive/yolo_runs/train_v3_5cls/weights/best.pt')
r = m.val(data=str(D / 'v3_colab.yaml'), split='test')
for i, name in m.names.items():
    print(f'{name:12s} mAP50 {r.box.ap50[list(r.box.ap_class_index).index(i)]:.3f}' if i in r.box.ap_class_index else f'{name:12s} test에 없음')
print('최종 가중치: /content/drive/MyDrive/yolo_runs/train_v3_5cls/weights/best.pt')